# Phase 1 — Retrieval-Augmented NLP Embeddings (BGE-large)

**Problem with previous approach:** one flat vector per year/season → same number broadcast to every day → zero discriminative signal.

**This approach:** for each of the 2642 daily rows, we build a date-aware query and retrieve the **top-5 most relevant chunks** from the reef reports. we mean-pool only those 5 chunks → genuinely different 1024-dim vector per day.

**Output:** `nlp_embeddings_daily.csv` — 2642 rows × (date, split, label, bge_0 … bge_1023)

**Estimated runtime:** 25–35 min on Colab (BGE encoding ~2642 queries)

---
## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install pymupdf sentence-transformers pytesseract Pillow pandas numpy tqdm
!apt-get install -y tesseract-ocr > /dev/null 2>&1
print('✅ Done')

---
## Step 2 — Upload files

In [ ]:
from google.colab import files

print('Upload the 3 ZIP files: AIMS_Reports_GBR.zip, MMP_Water_Quality.zip, Reef_Updates_Reports.zip')
print('You can select all 3 at once.')
uploaded_zips = files.upload()
for f in uploaded_zips:
    print(f'  ✅ {f}  ({len(uploaded_zips[f]):,} bytes)')

In [ ]:
print('Upload CNN embeddings CSV: cnn_embeddings_daily.csv')
uploaded_csv = files.upload()
for f in uploaded_csv:
    print(f'  ✅ {f}  ({len(uploaded_csv[f]):,} bytes)')

---
## Step 3 — Extract ZIPs and load CNN embeddings

In [ ]:
import zipfile, re, os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR   = Path('reef_data')
OUTPUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCE_MAP = {
    'AIMS_Reports_GBR.zip':    'AIMS',
    'MMP_Water_Quality.zip':   'MMP',
    'Reef_Updates_Reports.zip':'Reef',
}

# Extract ZIPs
for zip_name, key in SOURCE_MAP.items():
    if zip_name not in uploaded_zips:
        print(f'  ⚠️  {zip_name} not found'); continue
    out = DATA_DIR / key
    out.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(out)
    print(f'  ✅ {zip_name} → {out}')

all_pdfs = list(DATA_DIR.rglob('*.pdf'))
print(f'\nTotal PDFs: {len(all_pdfs)}')

# Load CNN embeddings — we only need date/split/label
csv_name = [f for f in uploaded_csv if f.endswith('.csv')][0]
cnn_df   = pd.read_csv(csv_name)
cnn_df['date']   = pd.to_datetime(cnn_df['date'])
cnn_df['year']   = cnn_df['date'].dt.year
cnn_df['month']  = cnn_df['date'].dt.month

def get_season(month):
    # Southern hemisphere (Australia)
    if month in [12, 1, 2]:  return 'Summer'
    elif month in [3, 4, 5]: return 'Autumn'
    elif month in [6, 7, 8]: return 'Winter'
    else:                    return 'Spring'

cnn_df['season'] = cnn_df['month'].apply(get_season)
print(f'\nCNN embeddings: {cnn_df.shape}')
print(f'Date range: {cnn_df["date"].min().date()} → {cnn_df["date"].max().date()}')
print(f'Unique year-season combos: {cnn_df.groupby(["year","season"]).ngroups}')

---
## Step 4 — Extract text from all PDFs and build chunk corpus

In [ ]:
import fitz
import pytesseract
from PIL import Image

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
MIN_CHUNK_LEN = 60

def clean_text(text):
    text = re.sub(r'[^\x09\x0A\x20-\x7E\x80-\xFF]', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    lines = [l for l in text.splitlines()
             if not re.fullmatch(r'\s*\d{1,4}\s*', l)]
    return '\n'.join(lines).strip()

def has_text_layer(doc, sample=8):
    n    = min(sample, len(doc))
    hits = sum(1 for i in range(n)
               if len(doc[i].get_text('text').strip()) > 40)
    return (hits / n) >= 0.15

def ocr_page(page, dpi=150):
    mat = fitz.Matrix(dpi/72, dpi/72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    return clean_text(pytesseract.image_to_string(img, config='--oem 3 --psm 6'))

def chunk_text(text):
    words  = text.split()
    chunks, i = [], 0
    while i < len(words):
        w = words[i: i + CHUNK_SIZE]
        if len(w) >= MIN_CHUNK_LEN:
            chunks.append(' '.join(w))
        i += (CHUNK_SIZE - CHUNK_OVERLAP)
    return chunks

def extract_year(name):
    m = re.search(r'20\d{2}', name)
    return int(m.group(0)) if m else None

# ── Build chunk corpus ────────────────────────────────────────────────────────
# Each chunk: {'text': str, 'year': int|None, 'source': str}
corpus = []

for pdf_path in tqdm(all_pdfs, desc='Extracting PDFs'):
    source = pdf_path.parts[-3] if len(pdf_path.parts) >= 3 else 'Unknown'
    year   = extract_year(pdf_path.name) or extract_year(str(pdf_path))
    doc    = fitz.open(str(pdf_path))

    if has_text_layer(doc):
        full_text = '\n\n'.join(clean_text(doc[i].get_text('text'))
                                for i in range(len(doc)))
        method = 'text_layer'
    else:
        print(f'  ⚙️  OCR: {pdf_path.name} ({len(doc)} pages)...')
        pages = []
        for i in tqdm(range(len(doc)), desc='OCR', leave=False):
            pages.append(ocr_page(doc[i]))
        full_text = '\n\n'.join(pages)
        method = 'ocr'

    doc.close()
    chunks = chunk_text(full_text)

    for chunk in chunks:
        corpus.append({'text': chunk, 'year': year, 'source': source})

    print(f'  ✅ {pdf_path.name[:55]:<55} year={year}  {len(chunks):>4} chunks  [{method}]')

print(f'\nTotal chunks in corpus: {len(corpus)}')

---
## Step 5 — Load BGE-large and embed entire corpus

We embed all chunks **once** upfront. retrieval per day is then just cosine similarity — fast.

In [ ]:
from sentence_transformers import SentenceTransformer

print('Loading BAAI/bge-large-en-v1.5 (~1.3GB)...')
bge = SentenceTransformer('BAAI/bge-large-en-v1.5')
BGE_DIM = bge.get_sentence_embedding_dimension()
print(f'✅ Loaded — dim: {BGE_DIM}')

In [ ]:
# BGE passage prefix for corpus chunks
PASSAGE_PREFIX = 'Represent this document for retrieval: '
QUERY_PREFIX   = 'Represent this sentence for searching relevant passages: '

corpus_texts = [PASSAGE_PREFIX + c['text'] for c in corpus]

print(f'Embedding {len(corpus_texts)} chunks...')
corpus_embs = bge.encode(
    corpus_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)  # shape: (n_chunks, 1024)

print(f'✅ Corpus embeddings: {corpus_embs.shape}')

# Save corpus embeddings in case Colab crashes
np.save(OUTPUT_DIR / 'corpus_embs.npy', corpus_embs)
print('  Checkpoint saved → corpus_embs.npy')

---
## Step 6 — Retrieval-augmented embedding per day

For each daily row we:
1. Build a **date-aware query**: `"coral bleaching thermal stress GBR {region} {season} {year}"`
2. Embed the query with BGE
3. Compute cosine similarity against all corpus chunks
4. Take **top-5** most relevant chunks
5. Mean-pool those 5 → 1024-dim vector for that day
6. Re-normalise

Result: every day gets a **genuinely different** embedding driven by what the reports say about that specific season/year.

In [ ]:
TOP_K = 5   # number of chunks to retrieve per day

# Pre-group unique (year, season) combos so we only encode
# one query per combo, not one per day (saves ~95% of encoding time)
unique_combos = cnn_df[['year','season']].drop_duplicates().reset_index(drop=True)
print(f'Unique (year, season) combos: {len(unique_combos)}')
print(unique_combos.to_string())

# Build queries for each unique combo
# Two queries per combo: one North, one Central — we average them
# since your model covers both regions
def build_queries(year, season):
    base = f'coral bleaching thermal stress Great Barrier Reef {season} {year}'
    return [
        QUERY_PREFIX + f'{base} Northern GBR reef condition DHW sea surface temperature',
        QUERY_PREFIX + f'{base} Central GBR inshore water quality bleaching alert',
    ]

# Embed all unique queries
print('\nEmbedding queries...')
combo_embeddings = {}   # (year, season) → 1024-dim

for _, row in tqdm(unique_combos.iterrows(), total=len(unique_combos), desc='Queries'):
    year, season = int(row['year']), row['season']
    queries      = build_queries(year, season)

    # Embed both region queries
    q_embs = bge.encode(
        queries,
        normalize_embeddings=True,
        show_progress_bar=False
    )  # (2, 1024)

    # Average North + Central queries → one query vector
    q_vec = q_embs.mean(axis=0)
    q_vec = q_vec / np.linalg.norm(q_vec)

    # Cosine similarity against entire corpus (dot product since L2-normalised)
    sims = corpus_embs @ q_vec   # (n_chunks,)

    # Top-K indices
    topk_idx = np.argsort(sims)[-TOP_K:][::-1]

    # Mean-pool top-K chunk embeddings
    topk_embs = corpus_embs[topk_idx]   # (K, 1024)
    pooled    = topk_embs.mean(axis=0)
    pooled    = pooled / np.linalg.norm(pooled)  # re-normalise

    combo_embeddings[(year, season)] = pooled

    # Show top retrieved chunk for first few combos
    if len(combo_embeddings) <= 4:
        best_chunk = corpus[topk_idx[0]]['text'][:120]
        print(f'  ({year}, {season:6s}) sim={sims[topk_idx[0]]:.3f} | {best_chunk}...')

print(f'\n✅ Combo embeddings computed: {len(combo_embeddings)}')

---
## Step 7 — Map combo embeddings to every daily row

In [ ]:
# Assign the right combo embedding to each daily row
daily_bge = np.zeros((len(cnn_df), BGE_DIM), dtype=np.float32)

for i, row in cnn_df.iterrows():
    key = (int(row['year']), row['season'])
    if key in combo_embeddings:
        daily_bge[i] = combo_embeddings[key]
    else:
        # Fallback: find nearest year with same season
        same_season = {k: v for k, v in combo_embeddings.items()
                       if k[1] == row['season']}
        nearest_year = min(same_season.keys(), key=lambda k: abs(k[0] - int(row['year'])))
        daily_bge[i] = same_season[nearest_year]
        print(f'  Fallback: {key} → used {nearest_year}')

print(f'daily_bge shape: {daily_bge.shape}')

# ── Sanity check: how many unique vectors do we have now? ────────────────────
# Round to 4dp to group near-identical vectors
rounded     = np.round(daily_bge, 4)
unique_rows = len(np.unique(rounded, axis=0))
print(f'Unique embedding vectors: {unique_rows} out of {len(daily_bge)}')
print(f'(Previous approach had ~7 unique vectors — this should be {len(unique_combos)})')

---
## Step 8 — Save NLP embeddings (standalone, NOT fused yet)

In [ ]:
import json, datetime

# ── Build output dataframe ───────────────────────────────────────────────────
bge_cols    = [f'bge_{i}' for i in range(BGE_DIM)]
bge_df      = pd.DataFrame(daily_bge, columns=bge_cols)
bge_df.insert(0, 'date',  cnn_df['date'].dt.strftime('%Y-%m-%d').values)
bge_df.insert(1, 'split', cnn_df['split'].values)
bge_df.insert(2, 'label', cnn_df['label'].values)
bge_df.insert(3, 'year',  cnn_df['year'].values)
bge_df.insert(4, 'season',cnn_df['season'].values)

# ── CSV ──────────────────────────────────────────────────────────────────────
csv_out = OUTPUT_DIR / 'nlp_embeddings_daily.csv'
bge_df.to_csv(csv_out, index=False)
print(f'✅ Saved CSV  : {csv_out}  {bge_df.shape}')

# ── NPY (features only) ──────────────────────────────────────────────────────
npy_out = OUTPUT_DIR / 'nlp_embeddings_daily.npy'
np.save(npy_out, daily_bge)
print(f'✅ Saved .npy : {npy_out}  {daily_bge.shape}')

# ── Metadata ─────────────────────────────────────────────────────────────────
meta = {
    'created':          datetime.datetime.now().isoformat(),
    'model':            'BAAI/bge-large-en-v1.5',
    'bge_dim':          BGE_DIM,
    'total_rows':       len(bge_df),
    'total_chunks':     len(corpus),
    'top_k_retrieval':  TOP_K,
    'unique_combos':    len(unique_combos),
    'unique_vectors':   int(unique_rows),
    'date_range':       [str(cnn_df['date'].min().date()),
                         str(cnn_df['date'].max().date())],
    'approach':         'retrieval-augmented per (year, season) combo',
    'query_template':   'coral bleaching thermal stress GBR {region} {season} {year}',
}
with open(OUTPUT_DIR / 'nlp_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✅ Saved meta : {OUTPUT_DIR}/nlp_metadata.json')

---
## Step 9 — Sanity check plots

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: unique vectors per season ────────────────────────────────────────
seasons   = ['Summer', 'Autumn', 'Winter', 'Spring']
s_counts  = [len([k for k in combo_embeddings if k[1]==s]) for s in seasons]
axes[0].bar(seasons, s_counts, color=['#FF6B6B','#FFD93D','#6BCBFF','#6BFF9E'])
axes[0].set_title('Unique BGE Vectors per Season')
axes[0].set_ylabel('Count')
for i, v in enumerate(s_counts):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# ── Plot 2: inter-combo cosine similarity heatmap ────────────────────────────
combos    = sorted(combo_embeddings.keys())
vec_mat   = np.stack([combo_embeddings[k] for k in combos])
sim_mat   = vec_mat @ vec_mat.T
labels    = [f"{y}\n{s[:3]}" for y, s in combos]
im = axes[1].imshow(sim_mat, cmap='Blues', vmin=0.7, vmax=1.0)
axes[1].set_xticks(range(len(combos))); axes[1].set_xticklabels(labels, fontsize=6)
axes[1].set_yticks(range(len(combos))); axes[1].set_yticklabels(labels, fontsize=6)
axes[1].set_title('Cosine Similarity Between\n(Year, Season) Combo Embeddings')
plt.colorbar(im, ax=axes[1])

# ── Plot 3: PCA of daily embeddings coloured by label ────────────────────────
pca     = PCA(n_components=2, random_state=42)
proj    = pca.fit_transform(daily_bge)
labels_arr = cnn_df['label'].values
colours = ['#00B4D8','#90E0EF','#FFD166','#EF476F','#073B4C']
lnames  = ['No Stress','Watch','Warning','Alert L1','Alert L2']
for lbl in range(5):
    mask = labels_arr == lbl
    axes[2].scatter(proj[mask,0], proj[mask,1],
                    c=colours[lbl], label=lnames[lbl],
                    s=8, alpha=0.6)
axes[2].set_title('PCA of Daily NLP Embeddings\n(coloured by stress label)')
axes[2].legend(markerscale=2, fontsize=7)
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

plt.suptitle('Phase 1 — NLP Embedding Quality Check', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nlp_sanity_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sanity check saved')

---
## Step 10 — Download outputs

In [ ]:
print('Downloading nlp_embeddings_daily.csv ...')
files.download(str(OUTPUT_DIR / 'nlp_embeddings_daily.csv'))

print('Downloading nlp_embeddings_daily.npy ...')
files.download(str(OUTPUT_DIR / 'nlp_embeddings_daily.npy'))

print('Downloading nlp_metadata.json ...')
files.download(str(OUTPUT_DIR / 'nlp_metadata.json'))

print('Downloading nlp_sanity_check.png ...')
files.download(str(OUTPUT_DIR / 'nlp_sanity_check.png'))

print('✅ All downloads triggered')

---
## What to check before sharing results

Share these 3 numbers after running:

1. **Unique vectors** — printed in Step 7. should be equal to number of unique (year, season) combos, NOT 7
2. **Similarity heatmap** — should have visible variation (not all 0.98–1.00 like before)
3. **PCA plot** — ideally stress labels show some clustering separation

If those look good → we move to Phase 2 (fusion model).